In [ ]:
# 1. 导入必要的库
import os
os.environ["HF_ENDPOINT"] = "https://hub.opencsg.com/hf"
os.environ["HF_TOKEN"] = "35c19102XXXXXXXXXXXXXXXXXXXXXX"   #这里更改为你自己的token

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 检查 GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"当前设备: {device}")

In [ ]:
# 2. 加载原生模型 (Qwen-0.5B)
MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"

print(f"正在加载模型: {MODEL_NAME} ... ")

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# 加载模型 (使用 float16 节省显存)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    trust_remote_code=True
).eval() # 设置为评估模式

print("模型加载完成！")

In [ ]:
# 3. 定义生成函数
def generate_poetry(prompt, max_len=60, temp=0.7, top_k=40):
    """
    prompt: 输入的诗句开头
    max_len: 生成的最大 token 数
    temp: 温度 (越高越随机)
    """
    print(f"\n--- 输入提示: '{prompt}' ---")
    
    # 编码输入
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs.input_ids.shape[1]
    
    # 生成
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=max_len,
            do_sample=True,
            temperature=temp,
            top_k=top_k,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # 解码并只保留新生成的部分
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"模型输出:\n{full_text}")
    return full_text

In [ ]:
# 4. 开始测试 (对比用)

# 测试案例 1: 经典五言绝句开头
generate_poetry("床前明月光，", max_len=20, temp=0.8)

# 测试案例 2: 七言诗句开头
generate_poetry("两个黄鹂鸣翠柳，", max_len=30, temp=0.8)